# 安装 LangChain

安装 LangChain 软件包：

```bash
pip install -U langchain

uv add langchain

```

LangChain 提供与数百个 LLM 和其他数千个集成的集成。这些集成以独立提供商包的形式存在。例如：
````bash
# Installing the OpenAI integration
pip install -U langchain-openai

# Installing the Anthropic integration
pip install -U langchain-anthropic


# Installing the OpenAI integration
uv add langchain-openai

# Installing the Anthropic integration
uv add langchain-anthropic

````

##  其他
请查看“[集成](https://docs.langchain.com/oss/python/integrations/providers/overview)”选项卡，获取可用集成的完整列表。



## 快速入门
本快速入门指南将引导您在短短几分钟内从简单的设置过渡到功能齐全的 AI 代理。

## 构建一个基本代理
首先创建一个简单的智能体，它可以回答问题并调用工具。该智能体将使用 Claude Sonnet 4.5 作为其语言模型，一个基本的天气函数作为工具，以及一个简单的提示来引导其行为。



In [4]:
from dotenv import load_dotenv
import os
load_dotenv("/Users/a1-6/Documents/projects/DL/.env")


True

In [5]:
from langchain.agents import create_agent
from langchain_deepseek.chat_models import ChatDeepSeek
model = ChatDeepSeek(
            model="deepseek-chat",
            max_retries=2,
            api_key=os.environ.get("DEEPSEEK_API_KEY"),
            api_base="https://api.deepseek.ai/v1",
        )

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="deepseek-chat", # 可以传递字符串或直接实例化模型，这个实例化的模型需要符合langchain的模型接,需要继承BaseChatModel 
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='0093b203-4e81-4e72-9190-183019966f22'),
  AIMessage(content="I'll check the weather in San Francisco for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 158, 'total_tokens': 184, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 158}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '11920e36-8a88-4f86-825e-a34923f21465', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--5d892761-0f64-40eb-a7dc-17d33f50c5db-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_00_9JfuFGbXcOqJNqYMNkFxFMak', 'type': 'tool_call'}], usage_metadata={'input_tokens': 158, 'output_tokens': 26, 'tot

## 构建一个现实世界的代理
接下来，构建一个实用的天气预报代理程序，以演示关键的生产概念：
- 详细的系统提示有助于改善代理行为
- 创建可与外部数据集成的工具
- 一致响应的模型配置
- 结构化输出，结果可预测
- 用于类似聊天互动的对话记忆
- 创建并运行代理，创建一个功能齐全的代理。

让我们一步一步来：

Step 1:  

定义系统提示符

系统提示定义了代理的角色和行为。请确保提示内容具体明确且可操作：

```bash
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""
```

Step 2:  

创建工具

工具允许模型通过调用您定义的函数与外部系统交互。工具可以依赖于运行时上下文，也可以与代理的内存进行交互。请注意以下工具如何`get_user_location`使用运行时上下文：

```python
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

```


Step 3:  

配置您的模型

根据您的使用场景，设置合适的语言模型参数：

```python
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "claude-sonnet-4-5-20250929",
    temperature=0.5,
    timeout=10,
    max_tokens=1000
)
```


step 4:  

定义响应格式

如果您需要代理响应与特定模式匹配，则可以选择定义结构化响应格式。

```python
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

```


Step 5:  

添加内存

为智能体添加记忆功能，以便在交互过程中保持状态。这样，智能体就能记住之前的对话和上下文。

```python
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

```


Step 6:  
创建并运行代理

现在将所有组件组装到您的代理中并运行它！

```python
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

```



恭喜！您现在拥有一个可以执行以下操作的 AI 代理：
- 理解语境并记住对话
- 巧妙运用多种工具
- 以统一的格式提供结构化的回答。
- 通过上下文处理用户特定信息
- 在互动过程中保持对话状态

## 哲学


LangChain 的存在是为了成为使用 LLM 进行构建的最简单起点，同时兼具灵活性和生产就绪性。

LangChain秉持以下几个核心理念：
- 大型语言模型（LLM）是一项强大而卓越的新技术。
- 将LLM与外部数据源结合使用，效果会更好。
- LLM将改变未来应用的面貌。具体而言，未来的应用将越来越注重自主性。

这一转变仍处于非常早期的阶段。

虽然构建这些智能体应用程序的原型很容易，但要构建足够可靠、可以投入生产的智能体仍然非常困难。

LangChain 的核心关注点有两个：
- 1.我们希望让开发者能够使用最佳模型进行构建。

    不同的供应商提供不同的API，这些API具有不同的模型参数和消息格式。标准化这些模型输入和输出是核心目标，这使得开发人员能够轻松切换到最新的先进模型，从而避免被供应商锁定。

- 2.我们希望让用户能够轻松地使用模型来协调与其他数据和计算交互的更复杂的流程。

    模型不应仅仅用于文本生成，还应用于协调与其他数据交互的更复杂的流程。LangChain 可以轻松定义语言学习模型(LLM) 可动态使用的工具，并有助于解析和访问非结构化数据。